# STFlow — per-slide flow-matching denoiser

STFlow (Huang et al., *ICML 2025* Spotlight) models the
joint distribution of gene expression across the **whole slide**
with a spatial transformer + flow-matching denoiser. The
upstream code releases training scripts only; this tutorial
therefore runs the canonical "fit-on-your-reference,
predict-on-your-query" workflow on a single GPU in a few minutes.

Use STFlow when STPath's gene vocabulary doesn't cover your panel,
when your tissue is far out-of-distribution from the 17 STPath
organs, or when you want to capture cross-spot coherence in the
output.

## Environment & demo data

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

import omicverse as ov
import lazyslide as zs
ov.utils.ov_plot_set()

adata, wsi = ov.space.histo.load_breast()
ov.space.histo.tile(wsi, tile_px=224, mpp=0.5)
ov.space.histo.embed(wsi, model='ctranspath',
                     batch_size=16, num_workers=0)

## Fit a flow-matching denoiser on the reference, predict on the query

Training the per-slide denoiser dominates the runtime. The cell below uses 80 epochs and a 500-gene HVG panel; the underlying transformer code is auto-cloned from `Graph-and-Geometric-Learning/STFlow` on first use.

In [ ]:
pred = ov.space.histo.predict_expression(
    wsi, method='stflow',
    reference=adata,
    fm_backbone='ctranspath',
    n_epochs=80, n_layers=4,
    n_sample_steps=5,
)
pred

## Visualise

In [ ]:
import scanpy as sc
sc.pl.embedding(pred, basis='spatial',
                color=list(pred.var_names[:4]),
                cmap='magma', s=12, ncols=2, frameon=False)